# Milestone 6F: Multi-Source Distributed ETL Pipeline with Data Quality Validation

This notebook runs the end-to-end multi-source ETL pipeline, demonstrating:
1. **Multi-Source Extraction**: Ingesting banking transactions, OFAC sanctions lists, and FATF country risk indices.
2. **Schema Mapping & Standardization**: Unifying schemas, currencies (15 currencies to USD), and timestamps.
3. **Deterministic Validation**: Isolating records with null fields, negative amounts, self-transfers, or future timestamps.
4. **Consistency Verification**: Deduplication and sanctions list cross-referencing.
5. **Feature Enrichment**: Attaching jurisdiction risk ratings and transaction velocity indicators.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.etl_pipeline import create_spark_session, ETLPipeline

spark = create_spark_session('Databricks-M6F-ETL')

tx_path = '../data/processed/processed_df.csv'
if not os.path.exists(tx_path):
    tx_path = '../data/raw/HI-Small_Trans.csv'

sanctions_path = '../data/external/sanctions_list.csv'
country_path = '../data/external/country_risk_scores.csv'

pipeline = ETLPipeline(
    spark=spark,
    transactions_path=tx_path,
    sanctions_path=sanctions_path,
    country_risk_path=country_path,
    max_transaction_rows=50000,
)

result = pipeline.run()
report = pipeline.generate_quality_report()


## 1. Data Quality and Audit Report
Inspect the automated audit metrics generated by the ETL pipeline.


In [ ]:
print('=== DATA QUALITY AUDIT REPORT ===')
print('Ingestion Counts:', report['total_records_by_source'])
print('Rejection Breakdown:', report['records_rejected_by_rule'])
print('Duplicates Removed:', report['deduplication_removed'])
print('Cleaned Valid Transactions:', report['total_valid_records'])
print('Sanctions Matches:', report['sanctions_matches_found'])
print('Referential Integrity:', report['referential_integrity'])


## 2. Enriched Account Profiles
Review final enriched profiles containing sanctions tags and jurisdiction risk scores.


In [ ]:
result.enriched_accounts.select(
    'account', 'in_degree', 'out_degree', 'total_in', 'is_sanctioned',
    'jurisdiction_code', 'jurisdiction_risk_score', 'max_txns_per_hour', 'is_laundering'
).show(10, truncate=False)
spark.stop()
